# Kahneman Framing × TRIBE v2 — Crash-Proof Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/akifnu/DSprojects/blob/main/tribev2/notebooks/Kahneman_Framing_RCT.ipynb)

**Text-only RCT** — no audio, no gTTS.

### Before you run
1. **Runtime → Change runtime type → GPU** (A100 40 GB recommended; T4 may OOM)
2. Optional: Colab **Secrets** → add `HF_TOKEN` with your Hugging Face read token (LLaMA access)
3. **Runtime → Run all** (first run installs deps and restarts once automatically)

Progress is checkpointed to `/content/framing_rct_checkpoint.json` so a crash mid-run can resume.

In [ ]:
import os, subprocess, sys

MARKER = '/content/.tribev2_colab_ready_v5'
REQ_URL = 'https://raw.githubusercontent.com/akifnu/DSprojects/main/tribev2/requirements-colab.txt'
REPO_URL = 'https://github.com/akifnu/DSprojects.git'
REPO_DIR = '/content/DSprojects'

if not os.path.exists(MARKER):
    if not os.path.exists(REPO_DIR):
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', REQ_URL])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO_DIR}/tribev2'])
    open(MARKER, 'w').write('ok')
    import IPython
    IPython.get_ipython().kernel.do_shutdown(restart=True)

print('Dependencies ready (marker found).')

In [ ]:
import sys
sys.path.insert(0, '/content/DSprojects/tribev2/src')

from tribe_capabilities.colab import bootstrap_imports, login_huggingface, query_gpu
from tribe_capabilities.inference import apply_torch_compat_patches

apply_torch_compat_patches()
bootstrap_imports()
login_huggingface()

gpu = query_gpu()
print('GPU:', gpu.name, '| VRAM (GB):', gpu.vram_gb, '| CUDA:', gpu.cuda_available)
for warning in gpu.warnings:
    print('⚠', warning)
if not gpu.cuda_available:
    raise RuntimeError('Enable a GPU runtime before continuing.')

In [ ]:
# --- User knobs ---
MAX_SCENARIOS = 6          # start small; raise to 12 or 318 after a clean run
PEAK_TIMESTEP = 5          # hemodynamic lag (~5 TRs)
MAX_PREDICT_ATTEMPTS = 3   # retry transient CUDA/OOM errors
CACHE_DIR = '/content/tribe_cache'
CHECKPOINT = '/content/framing_rct_checkpoint.json'

USE_FULL_BANK = False      # True → download all 318 pairs from GitHub
print(f'MAX_SCENARIOS={MAX_SCENARIOS}, USE_FULL_BANK={USE_FULL_BANK}')

In [ ]:
from tribe_capabilities.colab import fetch_scenarios_json

EMBEDDED_PAIRS = [
  {"scenario_id": "asian_disease", "domain": "health",
   "gain_frame": "Program A will save 200 people for certain. Program B has a one-third probability that all 600 people will be saved.",
   "loss_frame": "Program C will result in 400 people dying for certain. Program D has a one-third probability that nobody will die."},
  {"scenario_id": "surgery", "domain": "health",
   "gain_frame": "The operation has a 90 percent success rate. Nine out of ten patients recover fully.",
   "loss_frame": "The operation has a 10 percent failure rate. One out of ten patients do not survive."},
  {"scenario_id": "money_wallet", "domain": "financial",
   "gain_frame": "You can keep 20 dollars for sure, or gamble fifty-fifty to keep 30 dollars or only 10 dollars.",
   "loss_frame": "You will lose 10 dollars for sure and keep 20, or gamble fifty-fifty to lose nothing and keep 30, or lose 20 and keep 10."},
  {"scenario_id": "credit_card", "domain": "financial",
   "gain_frame": "Paying cash gives you a 1 dollar discount compared to the credit card price.",
   "loss_frame": "Paying by credit card adds a 1 dollar surcharge compared to the cash price."},
  {"scenario_id": "employment", "domain": "economic",
   "gain_frame": "The new policy will help 80 percent of workers keep their jobs next year.",
   "loss_frame": "The new policy means 20 percent of workers will lose their jobs next year."},
  {"scenario_id": "beef", "domain": "consumer",
   "gain_frame": "The label says the ground beef is 75 percent lean.",
   "loss_frame": "The label says the ground beef is 25 percent fat."},
  {"scenario_id": "exam", "domain": "education",
   "gain_frame": "You answered 60 percent of the exam questions correctly.",
   "loss_frame": "You answered 40 percent of the exam questions incorrectly."},
  {"scenario_id": "vaccine", "domain": "health",
   "gain_frame": "The vaccine caused no serious side effects in 95 percent of recipients.",
   "loss_frame": "The vaccine caused mild side effects in 5 percent of recipients."},
  {"scenario_id": "treatment_85", "domain": "health",
   "gain_frame": "This treatment works for 85 percent of patients.",
   "loss_frame": "This treatment fails for 15 percent of patients."},
  {"scenario_id": "investment", "domain": "financial",
   "gain_frame": "The fund gained value on 70 percent of trading days last year.",
   "loss_frame": "The fund lost value on 30 percent of trading days last year."},
  {"scenario_id": "pollution", "domain": "environment",
   "gain_frame": "The cleanup plan removes 40 percent of river pollution within five years.",
   "loss_frame": "The cleanup plan leaves 60 percent of river pollution in place within five years."},
  {"scenario_id": "course_pass", "domain": "education",
   "gain_frame": "72 percent of students passed the certification course on the first attempt.",
   "loss_frame": "28 percent of students failed the certification course on the first attempt."},
]

if USE_FULL_BANK:
    payload = fetch_scenarios_json()
    scenarios = payload['scenarios'][:MAX_SCENARIOS]
    print(f'Loaded {len(scenarios)} / {payload.get("n_scenarios", "?")} pairs from GitHub')
else:
    scenarios = EMBEDDED_PAIRS[:MAX_SCENARIOS]
    print(f'Using {len(scenarios)} embedded pairs (no network fetch)')

In [ ]:
import os
from pathlib import Path

from tribe_capabilities.config import TribeCapabilitiesConfig
from tribe_capabilities.inference import clear_cuda_cache, load_model, preload_llama_weights

os.makedirs(CACHE_DIR, exist_ok=True)
clear_cuda_cache()

config = TribeCapabilitiesConfig.load(Path('/content/DSprojects/tribev2/config/default.yaml'))
print('Preloading LLaMA weights (one-time, may take a few minutes)...')
preload_llama_weights(config)

print('Loading TRIBE v2 checkpoint...')
model = load_model(config, device='cuda')
print('Model ready.')

In [ ]:
import json
import numpy as np
from pathlib import Path

from tribe_capabilities.inference import predict_from_text_resilient, clear_cuda_cache


def summarize(preds: np.ndarray, t: int = PEAK_TIMESTEP) -> dict:
    t = min(t, preds.shape[0] - 1)
    return {
        'timesteps': int(preds.shape[0]),
        'mean_abs': float(np.mean(np.abs(preds))),
        'peak_abs': float(np.mean(np.abs(preds[t]))),
    }


def load_ckpt() -> dict:
    path = Path(CHECKPOINT)
    if not path.exists():
        return {'completed': {}, 'errors': []}
    return json.loads(path.read_text())


def save_ckpt(payload: dict) -> None:
    Path(CHECKPOINT).write_text(json.dumps(payload, indent=2))


ckpt = load_ckpt()
results = []

for scenario in scenarios:
    row = {'id': scenario['scenario_id'], 'domain': scenario.get('domain', '')}
    for frame in ('gain', 'loss'):
        key = f"{scenario['scenario_id']}_{frame}"
        text = scenario[f'{frame}_frame']
        if key in ckpt['completed']:
            stats = ckpt['completed'][key]
            row[f'{frame}_mean_abs'] = stats['mean_abs']
            row[f'{frame}_peak_abs'] = stats['peak_abs']
            print(f'[cached] {key}')
            continue

        def on_retry(attempt, exc):
            print(f'  retry {attempt}/{MAX_PREDICT_ATTEMPTS} for {key}: {type(exc).__name__}: {exc}')

        try:
            preds = predict_from_text_resilient(
                model,
                text,
                max_attempts=MAX_PREDICT_ATTEMPTS,
                on_retry=on_retry,
            )
            stats = summarize(preds)
            ckpt['completed'][key] = stats
            save_ckpt(ckpt)
            row[f'{frame}_mean_abs'] = stats['mean_abs']
            row[f'{frame}_peak_abs'] = stats['peak_abs']
            print(f'[ok] {key} peak={stats["peak_abs"]:.4f}')
        except Exception as exc:
            ckpt['errors'].append({'key': key, 'error': f'{type(exc).__name__}: {exc}'})
            save_ckpt(ckpt)
            print(f'[FAIL] {key}: {exc}')
            raise
        finally:
            clear_cuda_cache()

    if 'gain_mean_abs' in row and 'loss_mean_abs' in row:
        row['loss_minus_gain'] = row['loss_mean_abs'] - row['gain_mean_abs']
        results.append(row)
        print(f"{row['id']:20s}  loss-gain = {row['loss_minus_gain']:+.4f}")

print('Done:', len(results), 'pairs')

In [ ]:
import pandas as pd
from scipy import stats

if not results:
    raise RuntimeError('No completed pairs. Re-run inference cell or lower MAX_SCENARIOS.')

df = pd.DataFrame(results)
display(df[['id', 'domain', 'gain_mean_abs', 'loss_mean_abs', 'loss_minus_gain']])

gain_vals = df['gain_mean_abs'].values
loss_vals = df['loss_mean_abs'].values
t_stat, p_val = stats.ttest_rel(loss_vals, gain_vals)
diff = loss_vals - gain_vals
cohens_dz = diff.mean() / diff.std(ddof=1) if diff.std(ddof=1) > 0 else float('nan')
n_aligned = int((diff > 0).sum())

print('\n--- Kahneman framing test (text-only TRIBE path) ---')
print(f'Pairs: {len(df)}')
print(f'Loss > gain (mean |activation|): {n_aligned}/{len(df)} scenarios')
print(f'Mean difference (loss - gain): {diff.mean():.4f}')
print(f'Paired t-test p-value: {p_val:.4f}')
print(f"Cohen's dz: {cohens_dz:.3f}")
if diff.mean() > 0 and p_val < 0.05:
    print('\nDirection matches Kahneman loss-salience in this batch.')
else:
    print('\nNo significant Kahneman-aligned effect in this batch (try more pairs on A100).')